In [1]:
!pip install seqeval --quiet
!pip install pytorch-crf --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os, json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer
from seqeval.metrics import f1_score
from scipy.stats import wilcoxon

# ── Paths ──────────────────────────────────────────────────────────────────
CONSOLIDATED  = '/kaggle/input/datasets/michaelmarkey64/irish-ner-kg-consolidated'
ADKINS        = '/kaggle/input/datasets/michaelmarkey64/adkins-et-al-2025'
ABLATION      = '/kaggle/input/datasets/michaelmarkey64/irish-ner-ablation-results'
SRC           = '/kaggle/input/datasets/michaelmarkey64/irish-ner-kg-src'

TRAIN_CONLL   = f'{CONSOLIDATED}/data/conll/train_final.conll'
TEST_CONLL    = f'{CONSOLIDATED}/data/conll/NER_Irish_test.conll'

PHASE_A_PER   = f'{CONSOLIDATED}/data/kg/phase_a/per_nodes.csv'
PHASE_A_LOC   = f'{CONSOLIDATED}/data/kg/phase_a/loc_nodes.csv'
LOC_WIKIDATA  = f'{CONSOLIDATED}/loc_pool_wikidata.csv'
ORG_WIKIDATA  = f'{CONSOLIDATED}/org_pool_wikidata.csv'
ABLATION_JSON = f'{ABLATION}/ablation_results.json'

MODEL_NAME    = 'DCU-NLP/bert-base-irish-cased-v1'
SEEDS         = [42, 123, 256, 512, 999, 1024, 2048]
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')
print(f'Seeds:  {SEEDS}')

import sys
sys.path.insert(0, SRC)

Device: cuda
Seeds:  [42, 123, 256, 512, 999, 1024, 2048]


In [3]:
def load_conll(path):
    sentences, labels = [], []
    current_tokens, current_labels = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_labels)
                current_tokens, current_labels = [], []
            else:
                parts = line.split()
                current_tokens.append(parts[0])
                current_labels.append(parts[-1])
    if current_tokens:
        sentences.append(current_tokens)
        labels.append(current_labels)
    return sentences, labels

def get_train_entities(conll_path, entity_type):
    surfaces = set()
    current = []
    with open(conll_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                current = []
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            token, label = parts[0], parts[-1]
            if label == f'B-{entity_type}':
                current = [token]
            elif label == f'I-{entity_type}' and current:
                current.append(token)
            else:
                if current:
                    surfaces.add(' '.join(current))
                current = []
    return surfaces

train_sentences, train_labels = load_conll(TRAIN_CONLL)
test_sentences,  test_labels  = load_conll(TEST_CONLL)

print(f'Train sentences: {len(train_sentences)}')
print(f'Test sentences:  {len(test_sentences)}')

train_pers = get_train_entities(TRAIN_CONLL, 'PER')
train_locs = get_train_entities(TRAIN_CONLL, 'LOC')
train_orgs = get_train_entities(TRAIN_CONLL, 'ORG')

print(f'Train PER surfaces: {len(train_pers)}')
print(f'Train LOC surfaces: {len(train_locs)}')
print(f'Train ORG surfaces: {len(train_orgs)}')

Train sentences: 1006
Test sentences:  140
Train PER surfaces: 509
Train LOC surfaces: 473
Train ORG surfaces: 547


In [4]:
def is_clean(s):
    s = s.strip()
    if len(s) < 3:
        return False
    if s[0] in ("'", '"', '('):
        return False
    if all(not c.isalpha() for c in s):
        return False
    return True

# ── PER: Phase A Wikidata ──────────────────────────────────────────────────
per_df = pd.read_csv(PHASE_A_PER)
per_df['surface'] = per_df['label_ga'].fillna(per_df['label_en'])
all_pers   = per_df['surface'].dropna().str.strip().drop_duplicates().tolist()
novel_pers = [s for s in all_pers if is_clean(s) and s not in train_pers]

# ── LOC: Phase A + Wikidata enrichment ────────────────────────────────────
loc_a_df   = pd.read_csv(PHASE_A_LOC)
loc_a      = loc_a_df['canonical'].dropna().str.strip().drop_duplicates().tolist()

loc_w_df   = pd.read_csv(LOC_WIKIDATA)
loc_w_df   = loc_w_df[loc_w_df['label_ga'] != 'Ród']
loc_w      = loc_w_df['label_ga'].dropna().str.strip().drop_duplicates().tolist()

all_locs   = list(dict.fromkeys(loc_a + loc_w))
novel_locs = [s for s in all_locs if is_clean(s) and s not in train_locs]

# ── ORG: Wikidata enrichment ───────────────────────────────────────────────
org_w_df   = pd.read_csv(ORG_WIKIDATA)
all_orgs   = org_w_df['label_ga'].dropna().str.strip().drop_duplicates().tolist()
novel_orgs = [s for s in all_orgs if is_clean(s) and s not in train_orgs]

# ── Assemble pool ──────────────────────────────────────────────────────────
pool = {
    'PER': novel_pers,
    'LOC': novel_locs,
    'ORG': novel_orgs,
}

print('═' * 45)
print(f'{"Final Entity Pool Summary":^45}')
print('═' * 45)
for etype, surfaces in pool.items():
    print(f'{etype}: {len(surfaces)} novel surfaces')
print('═' * 45)

═════════════════════════════════════════════
          Final Entity Pool Summary          
═════════════════════════════════════════════
PER: 1256 novel surfaces
LOC: 584 novel surfaces
ORG: 346 novel surfaces
═════════════════════════════════════════════


In [5]:
def augment_conll(sentences, labels, pool, n_augments=1, seed=42):
    """
    RDA augmentation following Adkins et al.
    Finds O-tagged gaps in training sentences and inserts
    randomly selected typed entity spans from the KG pool
    with correct BIO labels. Returns original + augmented sentences.
    """
    random.seed(seed)
    all_sentences = list(zip(sentences, labels))

    for _ in range(n_augments):
        for tokens, lbls in zip(sentences, labels):

            # Find O-tagged gaps of length >= 2
            gaps = []
            i = 0
            while i < len(lbls):
                if lbls[i] == 'O':
                    start = i
                    while i < len(lbls) and lbls[i] == 'O':
                        i += 1
                    if (i - start) >= 2:
                        gaps.append((start, i))
                else:
                    i += 1

            if not gaps:
                continue

            gap_start, gap_end = random.choice(gaps)
            etype   = random.choice(['PER', 'LOC', 'ORG'])

            if not pool[etype]:
                continue

            surface       = random.choice(pool[etype])
            entity_tokens = surface.split()
            bio_labels    = [f'B-{etype}'] + [f'I-{etype}'] * (len(entity_tokens) - 1)

            new_tokens = list(tokens[:gap_start]) + entity_tokens + list(tokens[gap_start:])
            new_labels = list(lbls[:gap_start])   + bio_labels    + list(lbls[gap_start:])

            all_sentences.append((new_tokens, new_labels))

    return all_sentences


def save_conll(sentence_label_pairs, output_path):
    with open(output_path, 'w', encoding='utf-8') as f:
        for tokens, lbls in sentence_label_pairs:
            for token, label in zip(tokens, lbls):
                f.write(f'{token} {label}\n')
            f.write('\n')
    print(f'Saved {len(sentence_label_pairs)} sentences to {output_path}')


# Run augmentation 
augmented = augment_conll(train_sentences, train_labels, pool, n_augments=1, seed=42)
save_conll(augmented, '/kaggle/working/kg_rda_train.conll')

print(f'Original sentences:   {len(train_sentences)}')
print(f'Augmented sentences:  {len(augmented)}')

# Sanity check 
original_count = len(train_sentences)
print(f'\nSanity check — one augmented sentence:')
tokens, lbls = augmented[original_count]
for t, l in zip(tokens, lbls):
    print(f'  {t:30s} {l}')

Saved 2004 sentences to /kaggle/working/kg_rda_train.conll
Original sentences:   1006
Augmented sentences:  2004

Sanity check — one augmented sentence:
  ﻿Dáil                          B-ORG
  Éireann                        I-ORG
  Noel                           B-PER
  Hartnett                       I-PER
  06                             O
  /                              O
  07                             O
  /                              O
  2023                           O


In [6]:
from torchcrf import CRF
from transformers import BertModel

class BertCRF(nn.Module):
    """
    Standard gaBERT-CRF without KG injection.
    Used for KG-RDA experiment where the KG contribution
    is via data augmentation, not runtime embedding.
    """
    def __init__(self, num_tags, bert_dim=768):
        super().__init__()
        self.bert       = BertModel.from_pretrained(MODEL_NAME)
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_dim, num_tags)
        self.crf        = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        sequence  = self.bert(
                        input_ids=input_ids,
                        attention_mask=attention_mask
                    ).last_hidden_state
        emissions = self.classifier(self.dropout(sequence))
        mask      = attention_mask.bool()
        if labels is not None:
            labels = labels.clone()
            labels[labels == -100] = 0
            return -self.crf(emissions, labels, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask)


# Build label set from training data
def get_label_set(conll_path):
    labels = set()
    with open(conll_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                labels.add(parts[-1])
    return sorted(labels)

label_list = get_label_set(TRAIN_CONLL)
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
num_labels = len(label_list)

print(f'Labels ({num_labels}): {label_list}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME}')

Labels (7): ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded: DCU-NLP/bert-base-irish-cased-v1


In [7]:
from torch.utils.data import Dataset, DataLoader

class NERDataset(Dataset):
    def __init__(self, sentence_label_pairs, tokenizer, label2id, max_len=128):
        self.data      = sentence_label_pairs
        self.tokenizer = tokenizer
        self.label2id  = label2id
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids      = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        word_ids       = encoding.word_ids()

        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(self.label2id.get(labels[word_id], 0))
            else:
                label_ids.append(-100)
            prev_word_id = word_id

        label_ids = torch.tensor(label_ids, dtype=torch.long)

        return {
            'input_ids':      input_ids,
            'attention_mask': attention_mask,
            'labels':         label_ids
        }


def train_one_seed(seed, augmented_data, test_sentences, test_labels,
                   tokenizer, label2id, id2label, device,
                   epochs=5, batch_size=16, lr=3e-5):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_dataset = NERDataset(augmented_data, tokenizer, label2id)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model = BertCRF(num_tags=num_labels)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            loss = model(input_ids=input_ids,
                         attention_mask=attention_mask,
                         labels=labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        print(f'  Seed {seed} | Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}')

    # Evaluation 
    model.eval()
    all_preds, all_golds = [], []

    test_dataset = NERDataset(
        list(zip(test_sentences, test_labels)),
        tokenizer, label2id
    )
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            preds = model(input_ids=input_ids,
                          attention_mask=attention_mask)

            for pred_seq, label_seq in zip(preds, labels):
                pred_labels = []
                gold_labels = []
                for p, g in zip(pred_seq, label_seq):
                    if g.item() == -100:
                        continue
                    pred_labels.append(id2label[p])
                    gold_labels.append(id2label[g.item()])
                all_preds.append(pred_labels)
                all_golds.append(gold_labels)

    seed_f1 = f1_score(all_golds, all_preds)
    print(f'  Seed {seed} | Test F1: {seed_f1:.4f}')
    checkpoint_path = f'/kaggle/working/kg_rda_seed{seed}.pt'
    torch.save(model.state_dict(), checkpoint_path)
    print(f'  Saved checkpoint: {checkpoint_path}')
    return seed_f1


# Run across all seeds 
print('Starting KG-RDA training loop...')
print(f'Device: {DEVICE}')

kg_rda_f1_scores = []
for seed in SEEDS:
    print(f'\n── Seed {seed} ──')
    f1 = train_one_seed(
        seed=seed,
        augmented_data=augmented,
        test_sentences=test_sentences,
        test_labels=test_labels,
        tokenizer=tokenizer,
        label2id=label2id,
        id2label=id2label,
        device=DEVICE
    )
    kg_rda_f1_scores.append(f1)

print(f'\n{"═"*45}')
print(f'KG-RDA F1 scores: {[round(s, 4) for s in kg_rda_f1_scores]}')
print(f'Mean F1:          {np.mean(kg_rda_f1_scores):.4f}')
print(f'Std F1:           {np.std(kg_rda_f1_scores):.4f}')
print(f'{"═"*45}')

# Save results
kg_rda_results = {
    'condition':  'KG_RDA',
    'seeds':      SEEDS,
    'f1_scores':  kg_rda_f1_scores,
    'mean_f1':    float(np.mean(kg_rda_f1_scores)),
    'std_f1':     float(np.std(kg_rda_f1_scores))
}
with open('/kaggle/working/kg_rda_results.json', 'w') as f:
    json.dump(kg_rda_results, f, indent=2)
print('Saved kg_rda_results.json')

Starting KG-RDA training loop...
Device: cuda

── Seed 42 ──


pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DCU-NLP/bert-base-irish-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

Cancellation requested; stopping current tasks.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 112, in auto_conversion
    resolved_archive_file = cached_file(pretrained_model_name_or_path, filename, **cached_file_kwargs)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py", line 276, in cached_file
    file = cached_files(path_or_repo_id=path_or_repo_id, filenames=[filename], **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py"

In [ ]:
import subprocess
result = subprocess.run(['diff', 
    '/kaggle/input/datasets/michaelmarkey64/irish-ner-kg-consolidated/data/conll/train_final.conll',
    '/kaggle/input/datasets/michaelmarkey64/adkins-et-al-2025/train_final.conll'],
    capture_output=True, text=True)
print('Differences found:' if result.stdout else 'Files are identical')
print(result.stdout[:2000] if result.stdout else '')

In [ ]:
def check_bio_validity(conll_path):
    errors = 0
    sentences = 0
    prev_label = 'O'
    with open(conll_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                sentences += 1
                prev_label = 'O'
                continue
            parts = line.split()
            label = parts[-1]
            if label.startswith('I-'):
                entity_type = label[2:]
                if prev_label not in (f'B-{entity_type}', f'I-{entity_type}'):
                    errors += 1
            prev_label = label
    print(f'Sentences checked: {sentences}')
    print(f'BIO errors found:  {errors}')

check_bio_validity('/kaggle/working/kg_rda_train.conll')

In [ ]:
check_bio_validity('/kaggle/input/datasets/michaelmarkey64/irish-ner-kg-consolidated/data/conll/train_final.conll')

In [ ]:
# ── Baseline replication using clean training data ─────────────────────────
print('Starting baseline replication...')

baseline_sentences, baseline_labels = load_conll(TRAIN_CONLL)
baseline_data = list(zip(baseline_sentences, baseline_labels))

baseline_f1_scores = []
for seed in SEEDS:
    print(f'\n── Seed {seed} ──')
    f1 = train_one_seed(
        seed=seed,
        augmented_data=baseline_data,
        test_sentences=test_sentences,
        test_labels=test_labels,
        tokenizer=tokenizer,
        label2id=label2id,
        id2label=id2label,
        device=DEVICE
    )
    baseline_f1_scores.append(f1)

print(f'\n{"═"*45}')
print(f'Baseline F1 scores: {[round(s, 4) for s in baseline_f1_scores]}')
print(f'Mean F1:            {np.mean(baseline_f1_scores):.4f}')
print(f'Std F1:             {np.std(baseline_f1_scores):.4f}')
print(f'{"═"*45}')

In [ ]:
from scipy.stats import wilcoxon
import numpy as np

baseline  = [0.7765, 0.7627, 0.7635, 0.7522, 0.7528, 0.7415, 0.7568]
kg_rda    = [0.7169, 0.7564, 0.7327, 0.7536, 0.7608, 0.7422, 0.7447]

stat, p = wilcoxon(baseline, kg_rda)

print('═' * 50)
print(f'{"Wilcoxon Signed-Rank Test":^50}')
print('═' * 50)
print(f'Baseline mean F1:  {np.mean(baseline):.4f} ± {np.std(baseline):.4f}')
print(f'KG-RDA mean F1:    {np.mean(kg_rda):.4f} ± {np.std(kg_rda):.4f}')
print(f'Mean difference:   {np.mean(baseline) - np.mean(kg_rda):.4f}')
print(f'Statistic:         {stat:.4f}')
print(f'p-value:           {p:.4f}')
print(f'Significant:       {"Yes (p < 0.05)" if p < 0.05 else "No (p >= 0.05)"}')
print('═' * 50)

## Experiment Log

### Phase 1 — Entity Pool Construction
Built a three-type entity pool from external KG sources:
- PER: 1,256 novel surfaces (Phase A Wikidata per_nodes.csv)
- LOC: 584 novel surfaces (Phase A loc_nodes.csv + Wikidata enrichment)
- ORG: 346 novel surfaces (Wikidata enrichment)

All surfaces filtered to exclude entities already present in the Adkins training data.

### Phase 2 — Baseline Replication
Trained a standard gaBERT-CRF model across 7 seeds on the clean Adkins training data using the current training loop. 

**Result:** Mean F1 = 0.7580 ± 0.0102

This closely replicates Adkins et al.'s reported result of 0.7652, confirming the training loop is sound and providing a valid comparison point for subsequent experiments. A previously reported result of 0.8264 from an earlier notebook could not be replicated under these conditions and is not used as a comparison point.

### Phase 3 — KG-RDA Augmentation
Augmented the training data using the Adkins RDA mechanism, substituting the original self-referential entity pool with the externally sourced KG pool. This doubled the training set from 1,006 to 2,004 sentences by inserting KG entity surfaces into O-tagged gaps in existing training sentences.

**Result:** Mean F1 = 0.7439 ± 0.0141

Wilcoxon signed-rank test against the replicated baseline: p = 0.2969. The difference is not statistically significant. KG-informed RDA neither significantly improved nor significantly degraded performance relative to clean training data alone.

### Tentative Interpretation
The absence of a significant effect is consistent with at least two explanations that cannot currently be distinguished. First, the augmentation mechanism itself may be introducing noise — inserted entity surfaces in canonical nominative form are likely grammatically incoherent in many insertion positions in Irish, and training on malformed text at a 1:1 ratio with clean text may be disrupting the model's learning signal. Second, the pool content may simply not matter at this scale — the model may already generalise adequately from the clean training data and additional entity surfaces, however novel, do not shift the decision boundary in a meaningful way.

These explanations make different predictions and are in principle separable. A shuffled-entity control — augmenting with randomly typed entities from the pool — would distinguish mechanism noise from pool content effects. However, before committing to further training runs, a more informative preliminary step is to audit how many KG entity surfaces appear verbatim in the available unlabelled corpora (Herzog Dáil debates, ParlEE). If substantial verbatim coverage exists, corpus-mined naturally occurring sentences would be a more grammatically coherent augmentation source than synthetic insertion, and would directly address the morphological coherence problem identified above.

### Phase 4 — Corpus Coverage Audit
The following cell audits verbatim surface matches between the KG entity pools and the Herzog and ParlEE corpora. The purpose is to determine whether corpus-mined augmentation is viable before committing to another training run. A high match count would support Strategy 3 (corpus-mined entity contexts) as the most promising next step. A low match count would suggest the morphology ceiling is too severe for verbatim matching to be useful and would redirect attention toward inference-time strategies such as KG-constrained beam search or gazetteer-guided label smoothing.

In [ ]:
import pandas as pd
import re
from collections import defaultdict

HERZOG  = '/kaggle/input/datasets/michaelmarkey64/herzog-mikhaylov-dail-debates/Dail_debates_1919-2013/Dail_debates_1919-2013.tab'
PARLEE  = '/kaggle/input/datasets/michaelmarkey64/parlee-ie-plenary-speeches/ParlEE_IE_plenary_speeches.csv'

print('Loading Herzog corpus (sampled)...')
herzog_df   = pd.read_csv(HERZOG, sep='\t', on_bad_lines='skip', low_memory=False)
herzog_sample = herzog_df['speech'].dropna().astype(str).sample(n=5000, random_state=42)
herzog_text   = ' '.join(herzog_sample.tolist())
print(f'Herzog sample: {len(herzog_df)} speeches, using 5,000')
print(f'Herzog text length: {len(herzog_text):,} characters')

print('Loading ParlEE corpus (sampled)...')
parlee_df     = pd.read_csv(PARLEE, on_bad_lines='skip', low_memory=False)
parlee_sample = parlee_df['text'].dropna().astype(str).sample(n=5000, random_state=42)
parlee_text   = ' '.join(parlee_sample.tolist())
print(f'ParlEE sample: {len(parlee_df)} rows, using 5,000')
print(f'ParlEE text length: {len(parlee_text):,} characters')

combined_text = herzog_text + ' ' + parlee_text
print(f'Combined text length: {len(combined_text):,} characters')

In [ ]:
import re

def audit_pool(pool_surfaces, corpus_text, entity_type):
    matched   = []
    unmatched = []
    for surface in pool_surfaces:
        if re.search(re.escape(surface), corpus_text):
            matched.append(surface)
        else:
            unmatched.append(surface)
    coverage = len(matched) / len(pool_surfaces) * 100 if pool_surfaces else 0
    print(f'\n{entity_type}:')
    print(f'  Pool size:        {len(pool_surfaces)}')
    print(f'  Matched:          {len(matched)} ({coverage:.1f}%)')
    print(f'  Unmatched:        {len(unmatched)}')
    print(f'  Sample matched:   {matched[:10]}')
    print(f'  Sample unmatched: {unmatched[:5]}')
    return matched, unmatched

print('═' * 50)
print(f'{"Corpus Coverage Audit":^50}')
print('═' * 50)

per_matched, per_unmatched = audit_pool(pool['PER'], combined_text, 'PER')
loc_matched, loc_unmatched = audit_pool(pool['LOC'], combined_text, 'LOC')
org_matched, org_unmatched = audit_pool(pool['ORG'], combined_text, 'ORG')

total_pool    = len(pool['PER']) + len(pool['LOC']) + len(pool['ORG'])
total_matched = len(per_matched) + len(loc_matched) + len(org_matched)

print(f'\n{"═" * 50}')
print(f'Total pool:     {total_pool}')
print(f'Total matched:  {total_matched} ({total_matched/total_pool*100:.1f}%)')
print(f'{"═" * 50}')

matched_df = pd.DataFrame(
    [(s, 'PER') for s in per_matched] +
    [(s, 'LOC') for s in loc_matched] +
    [(s, 'ORG') for s in org_matched],
    columns=['surface', 'entity_type']
)
matched_df.to_csv('/kaggle/working/corpus_matched_surfaces.csv', index=False)
print(f'Saved corpus_matched_surfaces.csv with {len(matched_df)} entries.')

In [8]:
import torch
import numpy as np
from seqeval.metrics import f1_score

CHECKPOINT_DIR = '/kaggle/working'

# ── Build gazetteer from pool ──────────────────────────────────────────────
# Maps surface string -> entity type
gazetteer = {}
for etype, surfaces in pool.items():
    for surface in surfaces:
        gazetteer[surface] = etype

print(f'Gazetteer entries: {len(gazetteer)}')

# ── Build label mappings for gazetteer constraint ─────────────────────────
# We need to know which label indices correspond to B- and I- tags
b_tags = {etype: label2id[f'B-{etype}'] for etype in ['PER', 'LOC', 'ORG'] if f'B-{etype}' in label2id}
i_tags = {etype: label2id[f'I-{etype}'] for etype in ['PER', 'LOC', 'ORG'] if f'I-{etype}' in label2id}

print(f'B-tag indices: {b_tags}')
print(f'I-tag indices: {i_tags}')

# ── Gazetteer-constrained decode ───────────────────────────────────────────
def apply_gazetteer_constraints(emissions, tokens, gazetteer, b_tags, i_tags, boost=5.0):
    """
    Boost emission scores for known KG surfaces before CRF decode.
    emissions: tensor of shape (seq_len, num_tags)
    tokens: list of original token strings for this sentence
    """
    emissions = emissions.clone()
    token_list = tokens

    for surface, etype in gazetteer.items():
        surface_tokens = surface.split()
        n = len(surface_tokens)

        for i in range(len(token_list) - n + 1):
            window = token_list[i:i+n]
            if window == surface_tokens:
                # Boost B- tag for first token
                if etype in b_tags:
                    emissions[i, b_tags[etype]] += boost
                # Boost I- tags for remaining tokens
                for j in range(1, n):
                    if etype in i_tags:
                        emissions[i+j, i_tags[etype]] += boost

    return emissions


# ── Evaluation with gazetteer constraints ─────────────────────────────────
def evaluate_with_gazetteer(model, test_sentences, test_labels,
                             tokenizer, label2id, id2label,
                             gazetteer, b_tags, i_tags,
                             device, boost=5.0):
    model.eval()
    all_preds, all_golds = [], []

    with torch.no_grad():
        for tokens, labels in zip(test_sentences, test_labels):
            encoding = tokenizer(
                tokens,
                is_split_into_words=True,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            input_ids      = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)
            word_ids       = encoding.word_ids()

            # Get emissions without CRF decode
            sequence  = model.bert(
                            input_ids=input_ids,
                            attention_mask=attention_mask
                        ).last_hidden_state
            emissions = model.classifier(model.dropout(sequence)).squeeze(0)

            # Apply gazetteer boost using original tokens
            emissions = apply_gazetteer_constraints(
                emissions, tokens, gazetteer, b_tags, i_tags, boost=boost
            )

            # CRF decode with boosted emissions
            mask  = attention_mask.bool()
            preds = model.crf.decode(emissions.unsqueeze(0), mask=mask)[0]

            pred_labels = []
            gold_labels = []
            prev_word_id = None
            for idx, word_id in enumerate(word_ids):
                if word_id is None or word_id == prev_word_id:
                    prev_word_id = word_id
                    continue
                pred_labels.append(id2label[preds[idx]])
                gold_labels.append(labels[word_id])
                prev_word_id = word_id

            all_preds.append(pred_labels)
            all_golds.append(gold_labels)

    return f1_score(all_golds, all_preds)


# ── Run across all seeds ───────────────────────────────────────────────────
print('\nRunning Strategy 5 — KG-constrained inference...')
print(f'Boost value: 5.0')

kg_constrained_f1 = []

for seed in SEEDS:
    checkpoint_path = f'{CHECKPOINT_DIR}/kg_rda_seed{seed}.pt'
    model = BertCRF(num_tags=num_labels)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.to(DEVICE)

    f1 = evaluate_with_gazetteer(
        model=model,
        test_sentences=test_sentences,
        test_labels=test_labels,
        tokenizer=tokenizer,
        label2id=label2id,
        id2label=id2label,
        gazetteer=gazetteer,
        b_tags=b_tags,
        i_tags=i_tags,
        device=DEVICE,
        boost=5.0
    )
    print(f'  Seed {seed} | F1: {f1:.4f}')
    kg_constrained_f1.append(f1)

print(f'\n{"═" * 50}')
print(f'{"Strategy 5 — KG-Constrained Inference":^50}')
print(f'{"═" * 50}')
print(f'F1 scores:  {[round(s, 4) for s in kg_constrained_f1]}')
print(f'Mean F1:    {np.mean(kg_constrained_f1):.4f}')
print(f'Std F1:     {np.std(kg_constrained_f1):.4f}')
print(f'{"═" * 50}')

# ── Wilcoxon vs KG-RDA ────────────────────────────────────────────────────
from scipy.stats import wilcoxon
kg_rda = [0.7169, 0.7564, 0.7327, 0.7536, 0.7608, 0.7422, 0.7447]
stat, p = wilcoxon(kg_constrained_f1, kg_rda)
print(f'\nWilcoxon vs KG-RDA:')
print(f'  Mean difference: {np.mean(kg_constrained_f1) - np.mean(kg_rda):.4f}')
print(f'  p-value:         {p:.4f}')
print(f'  Significant:     {"Yes (p < 0.05)" if p < 0.05 else "No (p >= 0.05)"}')

# ── Wilcoxon vs baseline ───────────────────────────────────────────────────
baseline = [0.7765, 0.7627, 0.7635, 0.7522, 0.7528, 0.7415, 0.7568]
stat2, p2 = wilcoxon(kg_constrained_f1, baseline)
print(f'\nWilcoxon vs baseline replication:')
print(f'  Mean difference: {np.mean(kg_constrained_f1) - np.mean(baseline):.4f}')
print(f'  p-value:         {p2:.4f}')
print(f'  Significant:     {"Yes (p < 0.05)" if p2 < 0.05 else "No (p >= 0.05)"}')

# ── Save ───────────────────────────────────────────────────────────────────
import json
results = {
    'condition':  'KG_constrained_inference',
    'boost':      5.0,
    'seeds':      SEEDS,
    'f1_scores':  kg_constrained_f1,
    'mean_f1':    float(np.mean(kg_constrained_f1)),
    'std_f1':     float(np.std(kg_constrained_f1))
}
with open('/kaggle/working/kg_constrained_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved kg_constrained_results.json')

Gazetteer entries: 2186
B-tag indices: {'PER': 2, 'LOC': 0, 'ORG': 1}
I-tag indices: {'PER': 5, 'LOC': 3, 'ORG': 4}

Running Strategy 5 — KG-constrained inference...
Boost value: 5.0


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DCU-NLP/bert-base-irish-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/kg_rda_seed42.pt'

In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL — Combined Strategy 1 + 2 + 5 Experiment
# Three conditions, one training run, full seven-seed evaluation
# ═══════════════════════════════════════════════════════════════════════════

import os, json, random, zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertModel
from torchcrf import CRF
from seqeval.metrics import f1_score
from scipy.stats import wilcoxon

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS     = [42, 123, 256, 512, 999, 1024, 2048]
BOOST     = 3.0
SAVE_DIR  = '/kaggle/working/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Boost value: {BOOST}')
print(f'Checkpoint dir: {SAVE_DIR}')

# ── Hardcoded baselines ────────────────────────────────────────────────────
BASELINE_F1  = [0.7765, 0.7627, 0.7635, 0.7522, 0.7528, 0.7415, 0.7568]
KG_RDA_F1    = [0.7169, 0.7564, 0.7327, 0.7536, 0.7608, 0.7422, 0.7447]

# ── Gazetteer ──────────────────────────────────────────────────────────────
gazetteer = {}
for etype, surfaces in pool.items():
    for surface in surfaces:
        gazetteer[surface] = etype
print(f'Gazetteer entries: {len(gazetteer)}')

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME = 'DCU-NLP/bert-base-irish-cased-v1'

class BertCRF(nn.Module):
    def __init__(self, num_tags, bert_dim=768):
        super().__init__()
        self.bert       = BertModel.from_pretrained(MODEL_NAME)
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_dim, num_tags)
        self.crf        = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        sequence  = self.bert(
                        input_ids=input_ids,
                        attention_mask=attention_mask
                    ).last_hidden_state
        emissions = self.classifier(self.dropout(sequence))
        mask      = attention_mask.bool()
        if labels is not None:
            labels = labels.clone()
            labels[labels == -100] = 0
            return -self.crf(emissions, labels, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask), emissions

# ── Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    def __init__(self, sentence_label_pairs, tokenizer, label2id, max_len=128):
        self.data      = sentence_label_pairs
        self.tokenizer = tokenizer
        self.label2id  = label2id
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids      = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        word_ids       = encoding.word_ids()

        label_ids    = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(self.label2id.get(labels[word_id], 0))
            else:
                label_ids.append(-100)
            prev_word_id = word_id

        return {
            'input_ids':      input_ids,
            'attention_mask': attention_mask,
            'labels':         torch.tensor(label_ids, dtype=torch.long),
            'tokens':         tokens
        }

# ── Gazetteer emission boost ───────────────────────────────────────────────
def boost_emissions(emissions, tokens, gazetteer, b_tags, i_tags, boost):
    emissions = emissions.clone()
    for surface, etype in gazetteer.items():
        surface_tokens = surface.split()
        n = len(surface_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == surface_tokens:
                if etype in b_tags:
                    emissions[i, b_tags[etype]] += boost
                for j in range(1, n):
                    if etype in i_tags:
                        emissions[i+j, i_tags[etype]] += boost
    return emissions

# ── Constrained augmentation (sentence-initial insertion only) ─────────────
def augment_constrained(sentences, labels, pool, n_augments=1, seed=42):
    random.seed(seed)
    all_sentences = list(zip(sentences, labels))

    for _ in range(n_augments):
        for tokens, lbls in zip(sentences, labels):
            # Only insert at sentence-initial position if it is O-tagged
            # and there are at least 2 O tokens at the start
            if len(lbls) < 4:
                continue
            if lbls[0] != 'O' or lbls[1] != 'O':
                continue

            etype   = random.choice(['PER', 'LOC', 'ORG'])
            if not pool[etype]:
                continue
            surface       = random.choice(pool[etype])
            entity_tokens = surface.split()
            bio_labels    = [f'B-{etype}'] + [f'I-{etype}'] * (len(entity_tokens) - 1)

            new_tokens = entity_tokens + list(tokens)
            new_labels = bio_labels    + list(lbls)
            all_sentences.append((new_tokens, new_labels))

    return all_sentences

# ── Training and evaluation function ──────────────────────────────────────
def train_and_evaluate(seed, train_data, test_sentences, test_labels,
                       tokenizer, label2id, id2label, b_tags, i_tags,
                       device, condition_name,
                       use_gazetteer_train=False,
                       epochs=5, batch_size=16, lr=3e-5):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_dataset = NERDataset(train_data, tokenizer, label2id)
    def collate_fn(batch):
        return {
            'input_ids':      torch.stack([b['input_ids'] for b in batch]),
            'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
            'labels':         torch.stack([b['labels'] for b in batch]),
            'tokens':         [b['tokens'] for b in batch]
        }

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, collate_fn=collate_fn)

    model = BertCRF(num_tags=len(label2id))
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # ── Training ───────────────────────────────────────────────────────────
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            batch_tokens   = batch['tokens']

            if use_gazetteer_train:
                # Get emissions, apply boost, then compute CRF loss manually
                sequence  = model.bert(
                                input_ids=input_ids,
                                attention_mask=attention_mask
                            ).last_hidden_state
                emissions = model.classifier(model.dropout(sequence))
                mask      = attention_mask.bool()

                # Boost per sentence in batch
                
                boosted = []
                for i in range(len(batch_tokens)):
                    token_list = list(batch_tokens[i])
                    e = boost_emissions(
                        emissions[i], token_list,
                        gazetteer, b_tags, i_tags, BOOST
                    )
                    boosted.append(e)
                emissions = torch.stack(boosted)
                lbl = labels.clone()
                lbl[lbl == -100] = 0
                loss = -model.crf(emissions, lbl, mask=mask, reduction='mean')
            else:
                loss = model(input_ids=input_ids,
                             attention_mask=attention_mask,
                             labels=labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        print(f'  [{condition_name}] Seed {seed} | Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}')

    # ── Save checkpoint ────────────────────────────────────────────────────
    ckpt_path = f'{SAVE_DIR}/{condition_name}_seed{seed}.pt'
    torch.save(model.state_dict(), ckpt_path)

    # ── Evaluation ─────────────────────────────────────────────────────────
    model.eval()
    all_preds, all_golds = [], []

    with torch.no_grad():
        for tokens, lbls in zip(test_sentences, test_labels):
            encoding = tokenizer(
                tokens,
                is_split_into_words=True,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            input_ids      = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)
            word_ids       = encoding.word_ids()

            sequence  = model.bert(
                            input_ids=input_ids,
                            attention_mask=attention_mask
                        ).last_hidden_state
            emissions = model.classifier(model.dropout(sequence)).squeeze(0)
            mask      = attention_mask.bool()
            preds     = model.crf.decode(emissions.unsqueeze(0), mask=mask)[0]

            pred_labels, gold_labels = [], []
            prev_word_id = None
            for idx, word_id in enumerate(word_ids):
                if word_id is None or word_id == prev_word_id:
                    prev_word_id = word_id
                    continue
                pred_labels.append(id2label[preds[idx]])
                gold_labels.append(lbls[word_id])
                prev_word_id = word_id

            all_preds.append(pred_labels)
            all_golds.append(gold_labels)

    seed_f1 = f1_score(all_golds, all_preds)
    print(f'  [{condition_name}] Seed {seed} | Test F1: {seed_f1:.4f}')
    return seed_f1, model

# ── Inference-time gazetteer evaluation ───────────────────────────────────
def evaluate_with_gazetteer_inference(model, test_sentences, test_labels,
                                       tokenizer, label2id, id2label,
                                       b_tags, i_tags, device):
    model.eval()
    all_preds, all_golds = [], []

    with torch.no_grad():
        for tokens, lbls in zip(test_sentences, test_labels):
            encoding = tokenizer(
                tokens,
                is_split_into_words=True,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            input_ids      = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)
            word_ids       = encoding.word_ids()

            sequence  = model.bert(
                            input_ids=input_ids,
                            attention_mask=attention_mask
                        ).last_hidden_state
            emissions = model.classifier(model.dropout(sequence)).squeeze(0)

            # Apply gazetteer boost at inference
            emissions = boost_emissions(
                emissions, tokens, gazetteer, b_tags, i_tags, BOOST
            )

            mask  = attention_mask.bool()
            preds = model.crf.decode(emissions.unsqueeze(0), mask=mask)[0]

            pred_labels, gold_labels = [], []
            prev_word_id = None
            for idx, word_id in enumerate(word_ids):
                if word_id is None or word_id == prev_word_id:
                    prev_word_id = word_id
                    continue
                pred_labels.append(id2label[preds[idx]])
                gold_labels.append(lbls[word_id])
                prev_word_id = word_id

            all_preds.append(pred_labels)
            all_golds.append(gold_labels)

    return f1_score(all_golds, all_preds)

# ── Label tag indices for gazetteer ───────────────────────────────────────
b_tags = {e: label2id[f'B-{e}'] for e in ['PER','LOC','ORG'] if f'B-{e}' in label2id}
i_tags = {e: label2id[f'I-{e}'] for e in ['PER','LOC','ORG'] if f'I-{e}' in label2id}

# ── Build training data for each condition ─────────────────────────────────
clean_data       = list(zip(train_sentences, train_labels))
constrained_data = augment_constrained(train_sentences, train_labels, pool, n_augments=1, seed=42)

print(f'Condition A train size: {len(clean_data)} (clean)')
print(f'Condition B train size: {len(constrained_data)} (constrained augmented)')

# ══════════════════════════════════════════════════════════════════════════
# CONDITION A — Clean training + gazetteer boost at training time
# ══════════════════════════════════════════════════════════════════════════
print('\n' + '═'*55)
print('CONDITION A — Gazetteer boost, clean training data')
print('═'*55)

cond_a_f1     = []
cond_a_models = {}

for seed in SEEDS:
    print(f'\n── Seed {seed} ──')
    f1, model = train_and_evaluate(
        seed=seed,
        train_data=clean_data,
        test_sentences=test_sentences,
        test_labels=test_labels,
        tokenizer=tokenizer,
        label2id=label2id,
        id2label=id2label,
        b_tags=b_tags,
        i_tags=i_tags,
        device=DEVICE,
        condition_name='cond_a',
        use_gazetteer_train=True
    )
    cond_a_f1.append(f1)
    cond_a_models[seed] = model

print(f'\nCondition A — Mean F1: {np.mean(cond_a_f1):.4f} ± {np.std(cond_a_f1):.4f}')

# ══════════════════════════════════════════════════════════════════════════
# CONDITION B — Constrained augmentation + gazetteer boost at training time
# ══════════════════════════════════════════════════════════════════════════
print('\n' + '═'*55)
print('CONDITION B — Constrained augmentation + gazetteer boost')
print('═'*55)

cond_b_f1 = []

for seed in SEEDS:
    print(f'\n── Seed {seed} ──')
    f1, _ = train_and_evaluate(
        seed=seed,
        train_data=constrained_data,
        test_sentences=test_sentences,
        test_labels=test_labels,
        tokenizer=tokenizer,
        label2id=label2id,
        id2label=id2label,
        b_tags=b_tags,
        i_tags=i_tags,
        device=DEVICE,
        condition_name='cond_b',
        use_gazetteer_train=True
    )
    cond_b_f1.append(f1)

print(f'\nCondition B — Mean F1: {np.mean(cond_b_f1):.4f} ± {np.std(cond_b_f1):.4f}')

# ══════════════════════════════════════════════════════════════════════════
# CONDITION C — Inference-time gazetteer on Condition A checkpoints
# ══════════════════════════════════════════════════════════════════════════
print('\n' + '═'*55)
print('CONDITION C — Inference-time gazetteer (Condition A models)')
print('═'*55)

cond_c_f1 = []

for seed in SEEDS:
    model = cond_a_models[seed]
    f1 = evaluate_with_gazetteer_inference(
        model=model,
        test_sentences=test_sentences,
        test_labels=test_labels,
        tokenizer=tokenizer,
        label2id=label2id,
        id2label=id2label,
        b_tags=b_tags,
        i_tags=i_tags,
        device=DEVICE
    )
    print(f'  [cond_c] Seed {seed} | Test F1: {f1:.4f}')
    cond_c_f1.append(f1)

print(f'\nCondition C — Mean F1: {np.mean(cond_c_f1):.4f} ± {np.std(cond_c_f1):.4f}')

# ══════════════════════════════════════════════════════════════════════════
# STATISTICAL COMPARISON
# ══════════════════════════════════════════════════════════════════════════
print('\n' + '═'*55)
print(f'{"Full Results Summary":^55}')
print('═'*55)

conditions = {
    'Baseline replication': BASELINE_F1,
    'KG-RDA (previous)':    KG_RDA_F1,
    'Cond A (gazetteer train, clean)':    cond_a_f1,
    'Cond B (gazetteer train, augmented)': cond_b_f1,
    'Cond C (gazetteer inference)':        cond_c_f1,
}

for name, scores in conditions.items():
    print(f'{name[:40]:40s} Mean: {np.mean(scores):.4f}  Std: {np.std(scores):.4f}')

print('\nWilcoxon tests vs baseline replication:')
for name, scores in list(conditions.items())[2:]:
    stat, p = wilcoxon(scores, BASELINE_F1)
    sig = 'p < 0.05 *' if p < 0.05 else 'n.s.'
    print(f'  {name[:40]:40s} p={p:.4f}  {sig}')

# ══════════════════════════════════════════════════════════════════════════
# SAVE ALL RESULTS AND ZIP CHECKPOINTS
# ══════════════════════════════════════════════════════════════════════════
all_results = {
    'boost':         BOOST,
    'seeds':         SEEDS,
    'baseline':      {'f1': BASELINE_F1, 'mean': float(np.mean(BASELINE_F1))},
    'kg_rda':        {'f1': KG_RDA_F1,   'mean': float(np.mean(KG_RDA_F1))},
    'cond_a':        {'f1': cond_a_f1,   'mean': float(np.mean(cond_a_f1))},
    'cond_b':        {'f1': cond_b_f1,   'mean': float(np.mean(cond_b_f1))},
    'cond_c':        {'f1': cond_c_f1,   'mean': float(np.mean(cond_c_f1))},
}

with open('/kaggle/working/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('\nSaved all_results.json')

# Zip all checkpoints for safe download
zip_path = '/kaggle/working/checkpoints.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fname in os.listdir(SAVE_DIR):
        zf.write(os.path.join(SAVE_DIR, fname), fname)
print(f'Zipped checkpoints to {zip_path}')
print('\nDownload checkpoints.zip and all_results.json before session ends.')

Device: cuda
Boost value: 3.0
Checkpoint dir: /kaggle/working/checkpoints
Gazetteer entries: 2186
Condition A train size: 1006 (clean)
Condition B train size: 1892 (constrained augmented)

═══════════════════════════════════════════════════════
CONDITION A — Gazetteer boost, clean training data
═══════════════════════════════════════════════════════

── Seed 42 ──


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DCU-NLP/bert-base-irish-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [cond_a] Seed 42 | Epoch 1/5 | Loss: 18.5725
  [cond_a] Seed 42 | Epoch 2/5 | Loss: 4.5202
  [cond_a] Seed 42 | Epoch 3/5 | Loss: 2.4471
  [cond_a] Seed 42 | Epoch 4/5 | Loss: 1.3590
  [cond_a] Seed 42 | Epoch 5/5 | Loss: 0.8914
  [cond_a] Seed 42 | Test F1: 0.7615

── Seed 123 ──


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: DCU-NLP/bert-base-irish-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [cond_b] Seed 1024 | Epoch 1/5 | Loss: 12.6646
  [cond_b] Seed 1024 | Epoch 2/5 | Loss: 2.4951
  [cond_b] Seed 1024 | Epoch 3/5 | Loss: 1.0787
  [cond_b] Seed 1024 | Epoch 4/5 | Loss: 0.5289
  [cond_b] Seed 1024 | Epoch 5/5 | Loss: 0.3637
  [cond_b] Seed 1024 | Test F1: 0.7615

── Seed 2048 ──


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DCU-NLP/bert-base-irish-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [cond_b] Seed 2048 | Epoch 1/5 | Loss: 13.1600
  [cond_b] Seed 2048 | Epoch 2/5 | Loss: 2.4918
  [cond_b] Seed 2048 | Epoch 3/5 | Loss: 0.9978
  [cond_b] Seed 2048 | Epoch 4/5 | Loss: 0.5519
  [cond_b] Seed 2048 | Epoch 5/5 | Loss: 0.3254
  [cond_b] Seed 2048 | Test F1: 0.7496

Condition B — Mean F1: 0.7525 ± 0.0128

═══════════════════════════════════════════════════════
CONDITION C — Inference-time gazetteer (Condition A models)
═══════════════════════════════════════════════════════
  [cond_c] Seed 42 | Test F1: 0.7615
  [cond_c] Seed 123 | Test F1: 0.7283
  [cond_c] Seed 256 | Test F1: 0.7414
  [cond_c] Seed 512 | Test F1: 0.7670
  [cond_c] Seed 999 | Test F1: 0.7454
  [cond_c] Seed 1024 | Test F1: 0.7638
  [cond_c] Seed 2048 | Test F1: 0.7584

Condition C — Mean F1: 0.7523 ± 0.0132

═══════════════════════════════════════════════════════
                 Full Results Summary                  
═══════════════════════════════════════════════════════
Baseline replication           

In [13]:
import zipfile, os

SAVE_DIR = '/kaggle/working/checkpoints'
zip_path = '/kaggle/working/checkpoints_interim.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fname in os.listdir(SAVE_DIR):
        zf.write(os.path.join(SAVE_DIR, fname), fname)
print(f'Zipped {len(os.listdir(SAVE_DIR))} checkpoints to {zip_path}')

Zipped 14 checkpoints to /kaggle/working/checkpoints_interim.zip


In [14]:
import json
import numpy as np

summary = {
    'baseline':  {'f1': [0.7765, 0.7627, 0.7635, 0.7522, 0.7528, 0.7415, 0.7568], 'mean': 0.7580},
    'kg_rda':    {'f1': [0.7169, 0.7564, 0.7327, 0.7536, 0.7608, 0.7422, 0.7447], 'mean': 0.7439},
    'cond_a':    {'mean': 0.7540, 'std': 0.0127, 'p_vs_baseline': 0.5781},
    'cond_b':    {'mean': 0.7525, 'std': 0.0128, 'p_vs_baseline': 0.5781},
    'cond_c':    {'mean': 0.7523, 'std': 0.0132, 'p_vs_baseline': 0.5781},
}

print(json.dumps(summary, indent=2))

{
  "baseline": {
    "f1": [
      0.7765,
      0.7627,
      0.7635,
      0.7522,
      0.7528,
      0.7415,
      0.7568
    ],
    "mean": 0.758
  },
  "kg_rda": {
    "f1": [
      0.7169,
      0.7564,
      0.7327,
      0.7536,
      0.7608,
      0.7422,
      0.7447
    ],
    "mean": 0.7439
  },
  "cond_a": {
    "mean": 0.754,
    "std": 0.0127,
    "p_vs_baseline": 0.5781
  },
  "cond_b": {
    "mean": 0.7525,
    "std": 0.0128,
    "p_vs_baseline": 0.5781
  },
  "cond_c": {
    "mean": 0.7523,
    "std": 0.0132,
    "p_vs_baseline": 0.5781
  }
}
